# HeST Trace Event Viewer

Interactive viewer for pytesdaq-format HDF5 traces produced by `hest_trace_generation.py`.  
Browse events, inspect individual channel waveforms, view a 24-sensor energy heatmap, and overlay the original HeST truth data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.collections import PatchCollection
import matplotlib.colors as mcolors
import h5py
from pytesdaq.io import H5Reader
from ipywidgets import interact, IntSlider, Dropdown, Checkbox
%matplotlib inline

## Configuration

Set paths to the generated trace directory and (optionally) the original HeST H5 file.

In [ ]:
# --- USER CONFIG ---
TRACE_PATH = './generated/'  # path to pytesdaq output from hest_trace_generation.py
HEST_H5_PATH = None          # optional: path to original HeST .h5 for truth overlay
FS = 1.25e6                  # sampling frequency (Hz)
N_CHANNELS = 24

## Load traces

In [ ]:
reader = H5Reader()
reader.set_files(TRACE_PATH)

events = []
event_metadata = []
while True:
    traces, meta = reader.read_next_event(include_metadata=True, adctoamp=True)
    if meta['error_msg'] == 'No more files available':
        break
    events.append(traces.copy())
    event_metadata.append(meta)
reader.close()

n_events = len(events)
print(f'Loaded {n_events} events')
if n_events > 0:
    print(f'Trace shape: {events[0].shape}  ({events[0].shape[0]} channels, {events[0].shape[1]} samples)')

## Load HeST truth (optional)

In [ ]:
hest_data = None
if HEST_H5_PATH is not None:
    hest_data = h5py.File(HEST_H5_PATH, 'r')
    print(f'HeST file loaded: {hest_data.attrs["n_events"]} events, '
          f'{hest_data.attrs["n_sensors"]} sensors')
else:
    print('No HeST file specified (set HEST_H5_PATH to enable truth overlay)')

## HeRALD v1 sensor layout

The 24 sensors are arranged in a hex-packed 6x6 grid (with corners removed).  
This cell builds the (x, y) positions and grid indices for the heatmap display.

In [ ]:
ARRAY_MAP = np.array([[0,0,1,1,0,0],
                      [0,1,1,1,1,0],
                      [1,1,1,1,1,1],
                      [1,1,1,1,1,1],
                      [0,1,1,1,1,0],
                      [0,0,1,1,0,0]])
PITCH = 1.1  # cm

sensor_positions = {}  # sensor_id -> (x, y) in cm
sensor_grid = {}       # sensor_id -> (row, col) in array_map
sid = 0
for i in range(6):
    for j in range(6):
        if ARRAY_MAP[i, j] > 0.5:
            sensor_positions[sid] = ((i - 2.5) * PITCH, (j - 2.5) * PITCH)
            sensor_grid[sid] = (i, j)
            sid += 1

print(f'{len(sensor_positions)} sensors mapped')

## Helper functions

In [ ]:
def get_time_axis(n_samples, fs=FS):
    """Return time axis in milliseconds."""
    return np.arange(n_samples) / fs * 1e3


def get_pulse_amplitude(trace):
    """Estimate pulse amplitude as max - baseline (first 10% median)."""
    baseline_samples = max(1, len(trace) // 10)
    baseline = np.median(trace[:baseline_samples])
    return np.max(np.abs(trace - baseline))


def get_hest_truth(event_idx):
    """Extract per-sensor total energy from the HeST H5 for one event."""
    if hest_data is None:
        return None, None
    evt_key = str(event_idx)
    if evt_key not in hest_data['events']:
        return None, None
    evt = hest_data['events'][evt_key]
    recoil_energy = evt.attrs['recoil_energy']
    recoil_type = evt.attrs['recoil_type']
    sensor_energies = np.zeros(N_CHANNELS)
    for ch_key in ['singlet', 'triplet', 'ir', 'qp']:
        if ch_key not in evt:
            continue
        ch_grp = evt[ch_key]
        weight = ch_grp.attrs.get('weight', 1.0)
        for s in range(N_CHANNELS):
            s_key = str(s)
            if s_key in ch_grp:
                sensor_energies[s] += np.sum(ch_grp[s_key]['energies'][:]) * weight
    return {'recoil_energy': recoil_energy, 'recoil_type': recoil_type,
            'sensor_energies': sensor_energies}, None

## Plot: All 24 channels for a single event

In [ ]:
def plot_all_channels(event_idx=0):
    """Plot all 24 channel traces stacked vertically."""
    if event_idx >= n_events:
        print(f'Event {event_idx} out of range (max {n_events - 1})')
        return
    traces = events[event_idx]
    n_ch, n_samp = traces.shape
    t_ms = get_time_axis(n_samp)

    fig, axes = plt.subplots(6, 4, figsize=(16, 18), sharex=True)
    fig.suptitle(f'Event {event_idx} — All channels', fontsize=14)

    truth, _ = get_hest_truth(event_idx)
    if truth is not None:
        fig.suptitle(
            f'Event {event_idx} — {truth["recoil_type"]} recoil, '
            f'E = {truth["recoil_energy"]:.1f} eV',
            fontsize=14)

    for ch in range(min(n_ch, N_CHANNELS)):
        ax = axes[ch // 4, ch % 4]
        ax.plot(t_ms, traces[ch] * 1e6, linewidth=0.3, color='C0')
        ax.set_ylabel(f'ch {ch:02d}', fontsize=8)
        ax.tick_params(labelsize=7)
    axes[-1, 0].set_xlabel('Time (ms)')
    axes[-1, 1].set_xlabel('Time (ms)')
    axes[-1, 2].set_xlabel('Time (ms)')
    axes[-1, 3].set_xlabel('Time (ms)')
    fig.text(0.04, 0.5, 'Current (µA)', va='center', rotation='vertical', fontsize=12)
    plt.tight_layout(rect=[0.05, 0, 1, 0.97])
    plt.show()


interact(plot_all_channels,
         event_idx=IntSlider(min=0, max=max(0, n_events - 1), step=1, value=0,
                             description='Event'));

## Plot: Single channel detail

In [ ]:
channel_options = {f'channel_{i:02d}': i for i in range(N_CHANNELS)}

def plot_single_channel(event_idx=0, channel='channel_00'):
    """Detail view of a single channel with baseline and peak markers."""
    if event_idx >= n_events:
        print(f'Event {event_idx} out of range')
        return
    ch = channel_options[channel]
    trace = events[event_idx][ch]
    t_ms = get_time_axis(len(trace))

    baseline_n = max(1, len(trace) // 10)
    baseline = np.median(trace[:baseline_n])
    peak_idx = np.argmax(np.abs(trace - baseline))
    amplitude = trace[peak_idx] - baseline

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6),
                                   gridspec_kw={'height_ratios': [3, 1]})

    ax1.plot(t_ms, trace * 1e6, linewidth=0.5, color='C0')
    ax1.axhline(baseline * 1e6, color='gray', ls='--', lw=0.8, label='baseline')
    ax1.axvline(t_ms[peak_idx], color='red', ls=':', lw=0.8, label=f'peak @ {t_ms[peak_idx]:.2f} ms')
    ax1.set_ylabel('Current (µA)')
    ax1.set_title(f'Event {event_idx} — {channel}  |  amplitude = {amplitude*1e6:.4f} µA')
    ax1.legend(fontsize=8)

    # Zoomed view around peak
    zoom_half = max(1, int(0.002 * FS))  # +/- 2 ms around peak
    z_start = max(0, peak_idx - zoom_half)
    z_end = min(len(trace), peak_idx + zoom_half)
    ax2.plot(t_ms[z_start:z_end], trace[z_start:z_end] * 1e6,
             linewidth=0.8, color='C0')
    ax2.axhline(baseline * 1e6, color='gray', ls='--', lw=0.8)
    ax2.set_xlabel('Time (ms)')
    ax2.set_ylabel('Current (µA)')
    ax2.set_title('Zoomed (±2 ms around peak)', fontsize=10)

    plt.tight_layout()
    plt.show()


interact(plot_single_channel,
         event_idx=IntSlider(min=0, max=max(0, n_events - 1), step=1, value=0,
                             description='Event'),
         channel=Dropdown(options=list(channel_options.keys()), value='channel_00',
                          description='Channel'));

## Plot: Sensor heatmap

Shows the pulse amplitude (or integrated energy) on each sensor in the HeRALD v1 physical layout.

In [ ]:
def plot_sensor_heatmap(event_idx=0, use_truth=False):
    """Heatmap of per-sensor amplitude in the physical sensor layout."""
    if event_idx >= n_events:
        print(f'Event {event_idx} out of range')
        return

    if use_truth:
        truth, _ = get_hest_truth(event_idx)
        if truth is None:
            print('No HeST truth data available')
            return
        values = truth['sensor_energies']
        label = 'Deposited energy (eV)'
        title_extra = f' [TRUTH: {truth["recoil_type"]}, E={truth["recoil_energy"]:.1f} eV]'
    else:
        traces = events[event_idx]
        values = np.array([get_pulse_amplitude(traces[i]) for i in range(N_CHANNELS)])
        values *= 1e6  # to µA
        label = 'Pulse amplitude (µA)'
        title_extra = ''

    fig, ax = plt.subplots(1, 1, figsize=(7, 7))

    vmax = np.max(values) if np.max(values) > 0 else 1
    norm = mcolors.Normalize(vmin=0, vmax=vmax)
    cmap = plt.cm.inferno

    patches = []
    colors = []
    for sid in range(N_CHANNELS):
        x, y = sensor_positions[sid]
        rect = Rectangle((x - 0.5, y - 0.5), 1.0, 1.0)
        patches.append(rect)
        colors.append(values[sid])
        ax.text(x, y, f'{sid}', ha='center', va='center',
                fontsize=8, color='white' if values[sid] > vmax * 0.5 else 'black')

    pc = PatchCollection(patches, cmap=cmap, norm=norm)
    pc.set_array(np.array(colors))
    pc.set_edgecolor('gray')
    pc.set_linewidth(0.5)
    ax.add_collection(pc)

    ax.set_xlim(-3.5, 3.5)
    ax.set_ylim(-3.5, 3.5)
    ax.set_aspect('equal')
    ax.set_xlabel('x (cm)')
    ax.set_ylabel('y (cm)')
    ax.set_title(f'Event {event_idx} — Sensor map{title_extra}')

    cbar = fig.colorbar(pc, ax=ax, shrink=0.8)
    cbar.set_label(label)
    plt.tight_layout()
    plt.show()


interact(plot_sensor_heatmap,
         event_idx=IntSlider(min=0, max=max(0, n_events - 1), step=1, value=0,
                             description='Event'),
         use_truth=Checkbox(value=False, description='Show HeST truth'));

## Plot: Amplitude spectrum across events

Histogram of maximum pulse amplitudes across all events for a selected channel, useful for checking signal levels and noise floor.

In [ ]:
def plot_amplitude_spectrum(channel='channel_00', nbins=50):
    ch = channel_options[channel]
    amplitudes = np.array([get_pulse_amplitude(events[i][ch]) * 1e6
                           for i in range(n_events)])

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(amplitudes, bins=nbins, edgecolor='black', linewidth=0.5, alpha=0.8)
    ax.set_xlabel('Pulse amplitude (µA)')
    ax.set_ylabel('Count')
    ax.set_title(f'Amplitude distribution — {channel} ({n_events} events)')
    ax.axvline(np.median(amplitudes), color='red', ls='--',
               label=f'median = {np.median(amplitudes):.4f} µA')
    ax.legend()
    plt.tight_layout()
    plt.show()


interact(plot_amplitude_spectrum,
         channel=Dropdown(options=list(channel_options.keys()), value='channel_00',
                          description='Channel'));

## Plot: Total detected energy vs recoil energy (truth comparison)

Requires HeST H5 file. Shows the total energy deposited across all sensors vs the true recoil energy.

In [ ]:
if hest_data is not None:
    recoil_energies = []
    total_det_energies = []
    recoil_types = []

    for i in range(min(n_events, int(hest_data.attrs['n_events']))):
        truth, _ = get_hest_truth(i)
        if truth is not None:
            recoil_energies.append(truth['recoil_energy'])
            total_det_energies.append(np.sum(truth['sensor_energies']))
            recoil_types.append(truth['recoil_type'])

    recoil_energies = np.array(recoil_energies)
    total_det_energies = np.array(total_det_energies)

    fig, ax = plt.subplots(figsize=(8, 6))
    sc = ax.scatter(recoil_energies, total_det_energies, s=3, alpha=0.5)
    ax.plot([0, np.max(recoil_energies)], [0, np.max(recoil_energies)],
            'r--', lw=0.8, label='E_det = E_recoil')
    ax.set_xlabel('Recoil energy (eV)')
    ax.set_ylabel('Total detected energy (eV)')
    ax.set_title('Detection efficiency: total sensor energy vs recoil energy')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Set HEST_H5_PATH to enable this plot')

## Plot: Noise PSD estimate

Compute and display the power spectral density from a selected channel's traces, averaged over events.

In [ ]:
def plot_psd(channel='channel_00', n_avg=None):
    ch = channel_options[channel]
    if n_avg is None:
        n_avg = min(n_events, 50)

    n_samp = events[0].shape[1]
    freqs = np.fft.rfftfreq(n_samp, d=1.0 / FS)
    psd_avg = np.zeros(len(freqs))

    for i in range(n_avg):
        fft_vals = np.fft.rfft(events[i][ch])
        psd_avg += np.abs(fft_vals) ** 2
    psd_avg /= n_avg
    psd_avg *= 2.0 / (FS * n_samp)  # single-sided PSD normalization

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.loglog(freqs[1:], np.sqrt(psd_avg[1:]), linewidth=0.5)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel(r'ASD (A/$\sqrt{\mathrm{Hz}}$)')
    ax.set_title(f'Noise ASD — {channel} (averaged over {n_avg} events)')
    ax.grid(True, which='both', alpha=0.3)
    plt.tight_layout()
    plt.show()


interact(plot_psd,
         channel=Dropdown(options=list(channel_options.keys()), value='channel_00',
                          description='Channel'));

## Cleanup

In [ ]:
if hest_data is not None:
    hest_data.close()
    print('HeST file closed')